## Understand different bewteen 1 and 2-day model when inferring with FEV1 and bFEV1

In [1]:
import data.breathe_data as bd
import models.builders as mb
import pandas as pd
import inference.long_inf_slicing as slicing
import inference.helpers as ih
from plotly.subplots import make_subplots

In [ ]:
# HP: P(HFEV1|FEV1 or bFEV1) should be the same on 1-day and 2-day models
# Load the P(HFEV1|FEV1 and bFEV1) from the 1-day model
df1day = (
    bd.load_meas_from_excel(
        "infer_hfev1_priors_19_data_with_bFEV1_2016-19",
        study_folder="CFR",
        str_cols_to_arrays=[
            "P(HFEV1|FEV1)",
            "P(HFEV1|bFEV1)",
            # "P(HFEV1|FEF2575, bFEV1, FEV1)",
        ],
        use_csv=True,
        bypass_sanity_checks=True,
    )
)

# Calc the P(HFEV1|FEV1 and bFEV1) from the 2-day model

# Check if there are examples where P(HFEV1|FEV1) < P(HFEV1|bFEV1) in the 2-day model

# Check if P(HFEV1|FEV1 or bFEV1)_2day == P(HFEV1|FEV1 or bFEV1)_1day

## Archive



In [ ]:
df = bd.load_meas_from_excel(
    "pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
)

In [ ]:
diff_col = "ppFEV1FT - ppFEV1ST"
df = df.sort_values(by=[diff_col])

In [ ]:
df[df[diff_col] > 0][
    [
        "ecFEV1",
        "idx FEV1",
        "best FEV1",
        "idx best FEV1",
        diff_col,
        "FEV1%PredST",
        "FEV1%PredFT",
        "Age",
        "Height",
        "Sex",
    ]
][-10:-1].sort_values(by="ppFEV1FT - ppFEV1ST")

,ecFEV1,idx FEV1,best FEV1,idx best FEV1,ppFEV1FT - ppFEV1ST,FEV1%PredST,FEV1%PredFT,Age,Height,Sex
1832,1.58,31,1.58,31,1.389218,68.828957,70.218175,66,163,Female
1988,1.82,36,1.82,36,1.406492,62.156225,63.562717,61,165,Male
98,1.14,22,1.14,22,1.410682,52.337530,53.748212,64,158,Female
2014,1.40,28,1.45,29,1.448428,59.235603,60.684031,68,168,Female
526,0.97,19,1.02,20,1.461998,43.598415,45.060413,72,168,Female
1941,1.14,22,1.22,24,1.469845,55.911738,57.381583,72,161,Female
217,1.95,39,2.04,40,1.471929,63.113890,64.585818,69,176,Male
1801,1.15,23,1.27,25,1.548491,46.772008,48.320498,73,163,Male
1371,1.76,35,1.79,35,1.585664,58.821827,60.407492,67,172,Male


In [107]:
# Run inference on the 2-day model

idx = 1801
height = df.loc[idx, "Height"]
age = df.loc[idx, "Age"]
sex = df.loc[idx, "Sex"]

ecfev1_noise_model_cpt_suffix = "_std_add_mult_ecfev1"
ar_fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"
(
    _,
    inf_alg,
    HFEV1,
    uFEV1,
    ecFEV1,
    AR,
    ecFEF2575prctecFEV1,
) = mb.fev1_fef2575_point_in_time_model_noise_shared_healthy_vars(
    height,
    age,
    sex,
    ecfev1_noise_model_cpt_suffix=ecfev1_noise_model_cpt_suffix,
    ar_fef2575_cpt_suffix=ar_fef2575_cpt_suffix,
)

vars = [AR]
shared_vars = [HFEV1]
obs_vars = [ecFEV1.name]


ecfev1_obs_idx = df.loc[idx, f"idx FEV1"]
id = df.loc[idx, "ID"]

res = inf_alg.query(
    variables=[HFEV1.name],
    evidence={
        ecFEV1.name: ecfev1_obs_idx,
        # ecFEF2575prctecFEV1.name: ecfef2575prctecfev1_obs_idx,
    },
)

# res["Healthy FEV1 (L)"].values
HFEV1.get_mean(res["Healthy FEV1 (L)"].values)

2.4587355484294804

In [113]:
# Run inference on the 2-day model

ecfev1_obs_idx = df.loc[idx, f"idx best FEV1"]
id = df.loc[idx, "ID"]

res1 = inf_alg.query(
    variables=[HFEV1.name],
    evidence={
        ecFEV1.name: ecfev1_obs_idx,
        # ecFEF2575prctecFEV1.name: ecfef2575prctecfev1_obs_idx,
    },
)

# res["Healthy FEV1 (L)"].values
HFEV1.get_mean(res1["Healthy FEV1 (L)"].values)

2.462457730286084

In [108]:
# Run inference on the 2-day model with same FEV1 and best FEV1 values

vars = [AR]
shared_vars = [HFEV1]
obs_vars = [ecFEV1.name]


df_two_days = pd.DataFrame(
    [
        {
            "ID": df.loc[idx, "ID"],
            "Age": df.loc[idx, "Age"],
            "Sex": df.loc[idx, "Sex"],
            "Height": df.loc[idx, "Height"],
            "idx ecFEV1 (L)": df.loc[idx, "idx FEV1"],
            "ecFEV1": df.loc[idx, "ecFEV1"],
            "Date Recorded": df.loc[idx, "Date Recorded"],
        },
        {
            "ID": df.loc[idx, "ID"],
            "Age": df.loc[idx, "Age"],
            "Sex": df.loc[idx, "Sex"],
            "Height": df.loc[idx, "Height"],
            "idx ecFEV1 (L)": df.loc[idx, "idx best FEV1"],
            "ecFEV1": df.loc[idx, "best FEV1"],
            "Date Recorded": df.loc[idx, "Date Recorded"] + pd.Timedelta(days=1),
            # "Date Recorded": pd.to_datetime('2019-02-02').date(),
        },
    ]
)

df_query_res_two_days, _, _ = slicing.query_forwardly_across_days(
    df_two_days,
    inf_alg,
    shared_vars,
    vars,
    obs_vars,
    1e-8,
    [],
    debug=False,
)

In [109]:
df_query_res_two_days

,ID,Day,Age,Height,Sex,ecFEV1,Healthy FEV1 (L),Airway resistance (%)
0,B169198,2019-01-01,73,163,Male,1.15,"[6.64686094389978e-16, 3.9431287553119855e-11,...","[0.0001523856931666645, 0.0002997319004530737,..."
1,B169198,2019-01-02,73,163,Male,1.27,"[6.64686094389978e-16, 3.9431287553119855e-11,...","[0.0018919578842380709, 0.002218153140866074, ..."


In [115]:
fig = make_subplots(
    rows=3,
    cols=1,
)
print(HFEV1.get_mean(res["Healthy FEV1 (L)"].values))

ih.plot_histogram(
    fig, HFEV1, res["Healthy FEV1 (L)"].values, HFEV1.a, HFEV1.b, 1, 1, None, "#636EFA"
)

print(HFEV1.get_mean(res1["Healthy FEV1 (L)"].values))
ih.plot_histogram(
    fig, HFEV1, res1["Healthy FEV1 (L)"].values, HFEV1.a, HFEV1.b, 2, 1, None, "#636EFA"
)
print(HFEV1.get_mean(df_query_res_two_days.loc[0, "Healthy FEV1 (L)"]))
ih.plot_histogram(
    fig,
    HFEV1,
    df_query_res_two_days.loc[0, "Healthy FEV1 (L)"],
    HFEV1.a,
    HFEV1.b,
    3,
    1,
    None,
    "#636EFA",
)
fig.show()

2.4587355484294804
2.462457730286084
2.3799422946609305
